State 정의

In [7]:
from pydantic import BaseModel

class ChatState(BaseModel):
    query: str
    normalized_query: str | None = None
    answer: str | None = None

“LLM이 연동되었다고 가정한” mock 함수

In [8]:
def fake_llm_call(prompt: str) -> str:
    # 실제로는 여기서 LLM 호출
    return f"[LLM ANSWER]: {prompt}"

질문 정규화 node

In [9]:
def normalize_query(state: ChatState) -> ChatState:
    state.normalized_query = state.query.strip().lower()
    return state

답변 생성 node (LLM)

In [10]:
def generate_answer(state: ChatState) -> ChatState:
    state.answer = fake_llm_call(state.normalized_query)
    return state

Graph 생성

In [11]:
from langgraph.graph import StateGraph, END

graph_builder = StateGraph(ChatState)

# 노드 등록
graph_builder.add_node("normalize", normalize_query)
graph_builder.add_node("answer", generate_answer)

# 흐름 정의
graph_builder.set_entry_point("normalize")
graph_builder.add_edge("normalize", "answer")
graph_builder.add_edge("answer", END)

# 컴파일
graph = graph_builder.compile()

실행예

In [13]:
result = graph.invoke(
    ChatState(query="  LangGraph is COOL! ")
)

print(result)

{'query': '  LangGraph is COOL! ', 'normalized_query': 'langgraph is cool!', 'answer': '[LLM ANSWER]: langgraph is cool!'}
